In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
import warnings
warnings.filterwarnings('ignore')

print("Librerías cargadas correctamente.")

# Cargar dataset
daily = pd.read_csv("../data/processed/eurusd_con_sentimiento.csv",
                    index_col=0, parse_dates=True)

print(f"Dataset: {daily.shape}")

Librerías cargadas correctamente.
Dataset: (4031, 40)


In [5]:
features_ppo = [
    'momentum_5', 'momentum_10',
    'volatility_20', 'rsi_14', 'bb_position',
    'macd', 'sentiment_score', 'regime', 'trend_20_50', 'atr_14'
]

data_clean = daily[features_ppo + ['returns']].dropna().reset_index(drop=True)
env = KronosTradingEnv(data_clean, features_ppo)
check_env(env, warn=True)
print("Entorno verificado correctamente.")
print(f"Observaciones: {env.observation_space.shape}")
print(f"Acciones: {env.action_space.n} (HOLD, BUY, SELL)")

Entorno verificado correctamente.
Observaciones: (13,)
Acciones: 3 (HOLD, BUY, SELL)


In [7]:
print("Entrenando agente PPO...")

modelo_ppo = PPO(
    "MlpPolicy",
    env,
    learning_rate=3e-4,
    n_steps=512,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    verbose=0
)

modelo_ppo.learn(total_timesteps=50000)
print("Entrenamiento completado.")

obs, _ = env.reset()
rewards_totales = []
acciones = []
done = False

while not done:
    action, _ = modelo_ppo.predict(obs, deterministic=True)
    obs, reward, done, _, _ = env.step(action)
    rewards_totales.append(reward)
    acciones.append(action)

retorno_total = sum(rewards_totales)
acciones = np.array(acciones)
print(f"Retorno acumulado: {retorno_total:.4f}")
print(f"HOLD: {(acciones==0).sum()} ({(acciones==0).mean()*100:.1f}%)")
print(f"BUY:  {(acciones==1).sum()} ({(acciones==1).mean()*100:.1f}%)")
print(f"SELL: {(acciones==2).sum()} ({(acciones==2).mean()*100:.1f}%)")

Entrenando agente PPO...
Entrenamiento completado.
Retorno acumulado: -3.3630
HOLD: 0 (0.0%)
BUY:  3980 (99.3%)
SELL: 30 (0.7%)


In [8]:
# Resetear entorno y reentrenar con más timesteps
env2 = KronosTradingEnv(data_clean, features_ppo)

modelo_ppo2 = PPO(
    "MlpPolicy",
    env2,
    learning_rate=1e-4,
    n_steps=1024,
    batch_size=128,
    n_epochs=10,
    gamma=0.95,
    ent_coef=0.01,  # exploración
    verbose=0
)

print("Entrenando PPO con más exploración...")
modelo_ppo2.learn(total_timesteps=200000)
print("Completado.")

obs, _ = env2.reset()
rewards_totales2 = []
acciones2 = []
done = False

while not done:
    action, _ = modelo_ppo2.predict(obs, deterministic=True)
    obs, reward, done, _, _ = env2.step(action)
    rewards_totales2.append(reward)
    acciones2.append(action)

retorno2 = sum(rewards_totales2)
acciones2 = np.array(acciones2)
print(f"\nRetorno acumulado: {retorno2:.4f}")
print(f"HOLD: {(acciones2==0).sum()} ({(acciones2==0).mean()*100:.1f}%)")
print(f"BUY:  {(acciones2==1).sum()} ({(acciones2==1).mean()*100:.1f}%)")
print(f"SELL: {(acciones2==2).sum()} ({(acciones2==2).mean()*100:.1f}%)")

Entrenando PPO con más exploración...
Completado.

Retorno acumulado: 0.0400
HOLD: 0 (0.0%)
BUY:  81 (2.0%)
SELL: 3929 (98.0%)


In [9]:
# Guardar modelo PPO
modelo_ppo2.save("../data/processed/ppo_agent")
print("Agente PPO guardado.")

print(f"\nResumen Fase 9:")
print(f"  Entorno: KronosTradingEnv")
print(f"  Observaciones: 13 features")
print(f"  Acciones: HOLD, BUY, SELL")
print(f"  Timesteps: 200,000")
print(f"  Retorno acumulado: 0.04")
print(f"  Estado: prototipo funcional")
print(f"  Pendiente: reward shaping y más timesteps")

Agente PPO guardado.

Resumen Fase 9:
  Entorno: KronosTradingEnv
  Observaciones: 13 features
  Acciones: HOLD, BUY, SELL
  Timesteps: 200,000
  Retorno acumulado: 0.04
  Estado: prototipo funcional
  Pendiente: reward shaping y más timesteps
